In [ ]:
# Portable project paths. Set TLS_PROJECT_ROOT to the directory containing the input data.
import os
from pathlib import Path
PROJECT_ROOT = Path(os.environ.get("TLS_PROJECT_ROOT", ".")).resolve()


In [ ]:

import scanpy as sc
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

In [ ]:
adata = sc.read_h5ad("/data/beifen/zhongmin/slide-tag/H5AD格式数据/slide-tag肺_with_scanvi_加上补测数据_对3级淋巴结构进行分类_186_2025_12_30.h5ad")
print(adata)

In [ ]:
import re
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

h5ad_path = "/data/beifen/zhongmin/slide-tag/H5AD格式数据/slide-tag肺_with_scanvi_加上补测数据_对3级淋巴结构进行分类_186_2025_12_30.h5ad"

celltype_col = "celltype_4_ZZM"
tls_col = "tls_degrow_id_sample"
sample_col = "sample"

celltype_target = "CD4 Trm"
gene = "CCL5"
min_cells_each_group = 10

palette = {
    "non-TLS": "#57C3F3",
    "TLS": "#E95C59"
}

def normalize_label(x):
    x = str(x).strip()
    x = re.sub(r"[\s_\-]+", "", x)
    return x.lower()

adata = sc.read_h5ad(h5ad_path)

uniq_labels = pd.Series(adata.obs[celltype_col].astype(str).unique())
matched_labels = [x for x in uniq_labels if normalize_label(x) == normalize_label(celltype_target)]
if len(matched_labels) == 0:
    raise ValueError(
        f"在 {celltype_col} 中没有找到与 {celltype_target!r} 匹配的标签。\n"
        f"可先查看唯一值：\n{sorted(adata.obs[celltype_col].astype(str).unique())[:100]}"
    )

print("匹配到的细胞类型标签:", matched_labels)

sub = adata[adata.obs[celltype_col].astype(str).isin(matched_labels)].copy()
print(f"{celltype_target} cells: {sub.n_obs}")

sub.obs["group"] = np.where(
    sub.obs[tls_col].astype(str).str.fullmatch(r"S\d+_TLS\d+", na=False),
    "TLS",
    "non-TLS"
)

print("\nTLS / non-TLS 细胞数：")
print(sub.obs["group"].value_counts(dropna=False))

if gene not in sub.var_names:
    raise ValueError(f"{gene} 不在 adata.var_names 中。")

# 如果原始 counts 在 layers["counts"]，把下一句改成:
# X = sub.layers["counts"]
X = sub.X
gene_idx = sub.var_names.get_loc(gene)

if sp.issparse(X):
    gene_count = np.asarray(X[:, gene_idx].toarray()).ravel()
    libsize = np.asarray(X.sum(axis=1)).ravel()
else:
    gene_count = np.asarray(X[:, gene_idx]).ravel()
    libsize = np.asarray(X.sum(axis=1)).ravel()

cell_df = pd.DataFrame({
    "sample": sub.obs[sample_col].astype(str).values,
    "group": sub.obs["group"].values,
    "gene_count": gene_count,
    "libsize": libsize
})

# 每个 sample × group 单独统计细胞数
count_tbl = (
    cell_df.groupby(["sample", "group"], as_index=False)
    .size()
    .rename(columns={"size": "n_cells"})
)

print("\n每个 sample × group 的 CD4 Trm 细胞数：")
print(count_tbl)

# 只保留细胞数 >= 阈值的 sample × group
count_tbl2 = count_tbl[count_tbl["n_cells"] >= min_cells_each_group].copy()

if count_tbl2.empty:
    raise ValueError("没有任何 sample × group 满足细胞数阈值要求。")

print(f"\n满足阈值要求的 sample × group 数量: {count_tbl2.shape[0]}")
print(count_tbl2)

# 只保留这些合格的 sample × group
cell_df2 = cell_df.merge(
    count_tbl2[["sample", "group"]],
    on=["sample", "group"],
    how="inner"
)

# 做 pseudobulk：每个 sample × group 是 1 个点
pb = (
    cell_df2.groupby(["sample", "group"], as_index=False)
    .agg(
        gene_count_sum=("gene_count", "sum"),
        libsize_sum=("libsize", "sum"),
        n_cells=("sample", "size")
    )
)

pb["expr_pb"] = np.log1p(pb["gene_count_sum"] / pb["libsize_sum"] * 1e4)

print("\nPseudobulk table:")
print(pb)

x_non = pb.loc[pb["group"] == "non-TLS", "expr_pb"].values
x_tls = pb.loc[pb["group"] == "TLS", "expr_pb"].values

if len(x_non) < 2 or len(x_tls) < 2:
    raise ValueError("non-TLS 或 TLS 组的 pseudobulk 点数少于 2，无法进行非配对比较。")

stat, p = mannwhitneyu(x_non, x_tls, alternative="two-sided")

summary = (
    pb.groupby("group")["expr_pb"]
    .agg(["count", "mean", "std", "median"])
    .reindex(["non-TLS", "TLS"])
    .reset_index()
)

print("\nSummary:")
print(summary)
print(f"\nMann-Whitney U: P = {p:.3e}")

# =========================
# 作图：boxplot（四分位图）+ 散点
# 每个点 = 一个 sample × group 的 pseudobulk
# =========================
fig, ax = plt.subplots(figsize=(4.9, 6.1), dpi=300)

groups = ["non-TLS", "TLS"]
positions = [1, 2]
data = [
    pb.loc[pb["group"] == "non-TLS", "expr_pb"].values,
    pb.loc[pb["group"] == "TLS", "expr_pb"].values
]

bp = ax.boxplot(
    data,
    positions=positions,
    widths=0.5,
    patch_artist=True,
    showfliers=False,
    medianprops=dict(color="black", linewidth=1.6),
    boxprops=dict(edgecolor="black", linewidth=1.2),
    whiskerprops=dict(color="black", linewidth=1.2),
    capprops=dict(color="black", linewidth=1.2)
)

for patch, g in zip(bp["boxes"], groups):
    patch.set_facecolor(palette[g])

rng = np.random.default_rng(123)

for i, g in enumerate(groups):
    y = pb.loc[pb["group"] == g, "expr_pb"].values
    x = np.full(len(y), positions[i]) + rng.normal(0, 0.03, len(y))
    ax.scatter(
        x,
        y,
        s=38,
        color=palette[g],
        edgecolor="black",
        linewidth=0.45,
        zorder=3
    )

ax.set_xticks(positions)
ax.set_xticklabels([f"non-TLS\n{celltype_target}", f"TLS\n{celltype_target}"], fontsize=12)
ax.set_ylabel(f"log1p pseudobulk {gene} expression", fontsize=13)
ax.set_title(f"{gene} in {celltype_target}", fontsize=14)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(1.2)
ax.spines["bottom"].set_linewidth(1.2)
ax.tick_params(axis="both", width=1.2, length=5, labelsize=11)

ymax = max(pb["expr_pb"].max(), 0)
line_y = ymax * 1.08 + 0.02
h = ymax * 0.03 + 0.01
text_y = line_y + h + 0.01

ax.plot([1, 1, 2, 2], [line_y, line_y + h, line_y + h, line_y], lw=1.3, c="black")
p_text = "P < 1e-4" if p < 1e-4 else f"P = {p:.3e}"
ax.text(1.5, text_y, p_text, ha="center", va="bottom", fontsize=12)

ax.set_ylim(0, text_y * 1.12)

n_non = int(summary.loc[summary["group"] == "non-TLS", "count"].iloc[0])
n_tls = int(summary.loc[summary["group"] == "TLS", "count"].iloc[0])

ax.text(1, 0.02 * ax.get_ylim()[1], f"n={n_non}", ha="center", va="bottom", fontsize=10)
ax.text(2, 0.02 * ax.get_ylim()[1], f"n={n_tls}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()

out_prefix = f"{celltype_target.replace(' ', '_')}_sampleGroup_unpaired_pseudobulk_boxplot_{gene}"
fig.savefig(f"{out_prefix}.pdf", bbox_inches="tight", facecolor="white")
fig.savefig(f"{out_prefix}.png", bbox_inches="tight", facecolor="white")
pb.to_csv(f"{out_prefix}_pseudobulk_table.csv", index=False)
count_tbl.to_csv(f"{out_prefix}_sample_group_cell_count_table.csv", index=False)
summary.to_csv(f"{out_prefix}_summary.csv", index=False)

plt.show()

In [ ]:
import re
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

h5ad_path = "/data/beifen/zhongmin/slide-tag/H5AD格式数据/slide-tag肺_with_scanvi_加上补测数据_对3级淋巴结构进行分类_186_2025_12_30.h5ad"

celltype_col = "celltype_4_ZZM"
tls_col = "tls_degrow_id_sample"
sample_col = "sample"

celltype_target = "CD8 Trm"
gene = "CCL5"
min_cells_each_group = 10

palette = {
    "non-TLS": "#57C3F3",
    "TLS": "#E95C59"
}

def normalize_label(x):
    x = str(x).strip()
    x = re.sub(r"[\s_\-]+", "", x)
    return x.lower()

adata = sc.read_h5ad(h5ad_path)

uniq_labels = pd.Series(adata.obs[celltype_col].astype(str).unique())
matched_labels = [x for x in uniq_labels if normalize_label(x) == normalize_label(celltype_target)]
if len(matched_labels) == 0:
    raise ValueError(
        f"在 {celltype_col} 中没有找到与 {celltype_target!r} 匹配的标签。\n"
        f"可先查看唯一值：\n{sorted(adata.obs[celltype_col].astype(str).unique())[:100]}"
    )

print("匹配到的细胞类型标签:", matched_labels)

sub = adata[adata.obs[celltype_col].astype(str).isin(matched_labels)].copy()
print(f"{celltype_target} cells: {sub.n_obs}")

sub.obs["group"] = np.where(
    sub.obs[tls_col].astype(str).str.fullmatch(r"S\d+_TLS\d+", na=False),
    "TLS",
    "non-TLS"
)

print("\nTLS / non-TLS 细胞数：")
print(sub.obs["group"].value_counts(dropna=False))

if gene not in sub.var_names:
    raise ValueError(f"{gene} 不在 adata.var_names 中。")

# 如果原始 counts 在 layers["counts"]，把下一句改成:
# X = sub.layers["counts"]
X = sub.X
gene_idx = sub.var_names.get_loc(gene)

if sp.issparse(X):
    gene_count = np.asarray(X[:, gene_idx].toarray()).ravel()
    libsize = np.asarray(X.sum(axis=1)).ravel()
else:
    gene_count = np.asarray(X[:, gene_idx]).ravel()
    libsize = np.asarray(X.sum(axis=1)).ravel()

cell_df = pd.DataFrame({
    "sample": sub.obs[sample_col].astype(str).values,
    "group": sub.obs["group"].values,
    "gene_count": gene_count,
    "libsize": libsize
})

# 每个 sample × group 单独统计细胞数
count_tbl = (
    cell_df.groupby(["sample", "group"], as_index=False)
    .size()
    .rename(columns={"size": "n_cells"})
)

print("\n每个 sample × group 的 CD4 Trm 细胞数：")
print(count_tbl)

# 只保留细胞数 >= 阈值的 sample × group
count_tbl2 = count_tbl[count_tbl["n_cells"] >= min_cells_each_group].copy()

if count_tbl2.empty:
    raise ValueError("没有任何 sample × group 满足细胞数阈值要求。")

print(f"\n满足阈值要求的 sample × group 数量: {count_tbl2.shape[0]}")
print(count_tbl2)

# 只保留这些合格的 sample × group
cell_df2 = cell_df.merge(
    count_tbl2[["sample", "group"]],
    on=["sample", "group"],
    how="inner"
)

# 做 pseudobulk：每个 sample × group 是 1 个点
pb = (
    cell_df2.groupby(["sample", "group"], as_index=False)
    .agg(
        gene_count_sum=("gene_count", "sum"),
        libsize_sum=("libsize", "sum"),
        n_cells=("sample", "size")
    )
)

pb["expr_pb"] = np.log1p(pb["gene_count_sum"] / pb["libsize_sum"] * 1e4)

print("\nPseudobulk table:")
print(pb)

x_non = pb.loc[pb["group"] == "non-TLS", "expr_pb"].values
x_tls = pb.loc[pb["group"] == "TLS", "expr_pb"].values

if len(x_non) < 2 or len(x_tls) < 2:
    raise ValueError("non-TLS 或 TLS 组的 pseudobulk 点数少于 2，无法进行非配对比较。")

stat, p = mannwhitneyu(x_non, x_tls, alternative="two-sided")

summary = (
    pb.groupby("group")["expr_pb"]
    .agg(["count", "mean", "std", "median"])
    .reindex(["non-TLS", "TLS"])
    .reset_index()
)

print("\nSummary:")
print(summary)
print(f"\nMann-Whitney U: P = {p:.3e}")

# =========================
# 作图：boxplot（四分位图）+ 散点
# 每个点 = 一个 sample × group 的 pseudobulk
# =========================
fig, ax = plt.subplots(figsize=(4.9, 6.1), dpi=300)

groups = ["non-TLS", "TLS"]
positions = [1, 2]
data = [
    pb.loc[pb["group"] == "non-TLS", "expr_pb"].values,
    pb.loc[pb["group"] == "TLS", "expr_pb"].values
]

bp = ax.boxplot(
    data,
    positions=positions,
    widths=0.5,
    patch_artist=True,
    showfliers=False,
    medianprops=dict(color="black", linewidth=1.6),
    boxprops=dict(edgecolor="black", linewidth=1.2),
    whiskerprops=dict(color="black", linewidth=1.2),
    capprops=dict(color="black", linewidth=1.2)
)

for patch, g in zip(bp["boxes"], groups):
    patch.set_facecolor(palette[g])

rng = np.random.default_rng(123)

for i, g in enumerate(groups):
    y = pb.loc[pb["group"] == g, "expr_pb"].values
    x = np.full(len(y), positions[i]) + rng.normal(0, 0.03, len(y))
    ax.scatter(
        x,
        y,
        s=38,
        color=palette[g],
        edgecolor="black",
        linewidth=0.45,
        zorder=3
    )

ax.set_xticks(positions)
ax.set_xticklabels([f"non-TLS\n{celltype_target}", f"TLS\n{celltype_target}"], fontsize=12)
ax.set_ylabel(f"log1p pseudobulk {gene} expression", fontsize=13)
ax.set_title(f"{gene} in {celltype_target}", fontsize=14)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(1.2)
ax.spines["bottom"].set_linewidth(1.2)
ax.tick_params(axis="both", width=1.2, length=5, labelsize=11)

ymax = max(pb["expr_pb"].max(), 0)
line_y = ymax * 1.08 + 0.02
h = ymax * 0.03 + 0.01
text_y = line_y + h + 0.01

ax.plot([1, 1, 2, 2], [line_y, line_y + h, line_y + h, line_y], lw=1.3, c="black")
p_text = "P < 1e-4" if p < 1e-4 else f"P = {p:.3e}"
ax.text(1.5, text_y, p_text, ha="center", va="bottom", fontsize=12)

ax.set_ylim(0, text_y * 1.12)

n_non = int(summary.loc[summary["group"] == "non-TLS", "count"].iloc[0])
n_tls = int(summary.loc[summary["group"] == "TLS", "count"].iloc[0])

ax.text(1, 0.02 * ax.get_ylim()[1], f"n={n_non}", ha="center", va="bottom", fontsize=10)
ax.text(2, 0.02 * ax.get_ylim()[1], f"n={n_tls}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()

out_prefix = f"{celltype_target.replace(' ', '_')}_sampleGroup_unpaired_pseudobulk_boxplot_{gene}"
fig.savefig(f"{out_prefix}.pdf", bbox_inches="tight", facecolor="white")
fig.savefig(f"{out_prefix}.png", bbox_inches="tight", facecolor="white")
pb.to_csv(f"{out_prefix}_pseudobulk_table.csv", index=False)
count_tbl.to_csv(f"{out_prefix}_sample_group_cell_count_table.csv", index=False)
summary.to_csv(f"{out_prefix}_summary.csv", index=False)

plt.show()

In [ ]:
# 设置 Pandas 的显示选项，确保所有内容都能打印出来
pd.set_option('display.max_rows', None)  # 设置最大显示行数为 None，即显示所有行
pd.set_option('display.max_colwidth', None)  # 设置最大列宽为 None，即显示所有列内容

print(adata.obs['celltype_4_ZZM'].value_counts())

# 恢复默认的显示设置
pd.reset_option('display.max_rows')
pd.reset_option('display.max_colwidth')

In [ ]:
adata.obs['celltype_4_ZZM'] = adata.obs['celltype_4_ZZM'].astype(str)
adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'CTSL_Macrophage'), 'celltype_4_ZZM'] = 'Macrophages'
adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'CCSER1_Macrophage'), 'celltype_4_ZZM'] = 'Macrophages'
adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'SPARCL1_Macrophage'), 'celltype_4_ZZM'] = 'Macrophages'
adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'FYN_macrophages'), 'celltype_4_ZZM'] = 'Macrophages'
# adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'Monocyte'), 'celltype_4_ZZM'] = 'Macrophages'
adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'Macrophage_Cycling'), 'celltype_4_ZZM'] = 'Macrophages'
# adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'Alveolar macrophages'), 'celltype_4_ZZM'] = 'Macrophages'

adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'DOCK4_Fib'), 'celltype_4_ZZM'] = 'Fibroblasts'
adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'GSN_Fib'), 'celltype_4_ZZM'] = 'Fibroblasts'
adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'CD74_Fib'), 'celltype_4_ZZM'] = 'Fibroblasts'


adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'B naive'), 'celltype_4_ZZM'] = 'B cells'
adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'GCB'), 'celltype_4_ZZM'] = 'B cells'

# adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'CD4 Trm'), 'celltype_4_ZZM'] = 'CD4 T cells'
# adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'Tfh'), 'celltype_4_ZZM'] = 'CD4 T cells'
adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'CD4 T_Cycling'), 'celltype_4_ZZM'] = 'CD4 T cells'

# adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'CD8 Trm'), 'celltype_4_ZZM'] = 'CD8 T cells'

# adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'Tip cells'), 'celltype_4_ZZM'] = 'Endothelial cells'
# adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'Arterial ECs'), 'celltype_4_ZZM'] = 'Endothelial cells'
# adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'Lymphatic ECs'), 'celltype_4_ZZM'] = 'Endothelial cells'
# adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'Venous ECs'), 'celltype_4_ZZM'] = 'Endothelial cells'
# adata.obs.loc[(adata.obs['celltype_4_ZZM'] == 'Capillary ECs'), 'celltype_4_ZZM'] = 'Endothelial cells'

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


def plot_gene_in_selected_celltypes(
    adata,
    gene="CCL5",
    celltype_col="celltype_4_ZZM",
    selected_celltypes=None,
    counts_layer=None,
    target_sum=1e4,
    min_cells_per_type=20,
    order_by="expr_score_desc",
    max_points_per_type=800,
    figsize=(16, 7),
    save_prefix="immune_celltypes_gene_boxplot",
    title_fontsize=18,
    axis_label_fontsize=16,
    tick_fontsize=13,
    xtick_fontsize=12
):
    if selected_celltypes is None:
        selected_celltypes = [
            "Macrophages",
            "CD4 T cells", "CD8 T cells", "CD4 Trm", "CD8 Trm", "Treg", "Tfh", "NK cells", "CD4 T_Cycling",
            "B cells", "B naive", "GCB", "Plasma cells",
            "CTSL_Macrophage", "Alveolar macrophages", "CCSER1_Macrophage", "FYN_macrophages",
            "SPARCL1_Macrophage", "Macrophage_Cycling", "Monocyte",
            "DC1", "DC2", "LAMP3_DC", "Mast cells"
        ]

    palette = [
        '#E5D2DD', '#53A85F', '#F1BB72', '#F3B1A0', '#D6E7A3', '#57C3F3', '#476D87',
        '#E95C59', '#E59CC4', '#AB3282', '#23452F', '#BD956A', '#8C549C', '#585658',
        '#9FA3A8', '#E0D4CA', '#5F3D69', '#C5DEBA', '#58A4C3', '#E4C755', '#F7F398',
        '#AA9A59', '#E63863', '#E39A35', '#C1E6F3', '#6778AE', '#91D0BE', '#B53E2B',
        '#712820', '#DCC1DD', '#CCE0F5', '#CCC9E6', '#625D9E', '#68A180', '#3A6963',
        '#968175'
    ]

    if celltype_col not in adata.obs.columns:
        raise ValueError(f"{celltype_col} 不在 adata.obs.columns 中。")

    ad = adata.copy()

    ad.obs[celltype_col] = ad.obs[celltype_col].astype(str).str.strip()
    selected_celltypes = [str(x).strip() for x in selected_celltypes]

    if counts_layer is not None:
        if counts_layer not in ad.layers.keys():
            raise ValueError(f"{counts_layer} 不在 adata.layers 中。")
        ad.X = ad.layers[counts_layer].copy()

    if gene not in ad.var_names:
        raise ValueError(f"{gene} 不在 adata.var_names 中。")

    ad = ad[ad.obs[celltype_col].isin(selected_celltypes)].copy()

    if ad.n_obs == 0:
        raise ValueError("筛选后没有细胞，请检查 selected_celltypes 是否和数据中的标签一致。")

    sc.pp.normalize_total(ad, target_sum=target_sum)
    sc.pp.log1p(ad)

    expr = ad[:, gene].X
    if sp.issparse(expr):
        expr = np.asarray(expr.toarray()).ravel()
    else:
        expr = np.asarray(expr).ravel()

    df = pd.DataFrame({
        "celltype": ad.obs[celltype_col].values,
        "expr": expr
    })

    summary = (
        df.groupby("celltype")["expr"]
        .agg(["count", "mean", "median", "std"])
        .reset_index()
        .rename(columns={"count": "n_cells"})
    )

    pct_nonzero = (
        df.assign(nonzero=df["expr"] > 0)
        .groupby("celltype")["nonzero"]
        .mean()
        .reset_index()
        .rename(columns={"nonzero": "pct_nonzero"})
    )

    summary = summary.merge(pct_nonzero, on="celltype", how="left")
    summary["pct_nonzero"] = summary["pct_nonzero"] * 100
    summary["expr_score"] = summary["mean"] * summary["pct_nonzero"] / 100

    summary = summary[summary["n_cells"] >= min_cells_per_type].copy()

    if summary.shape[0] == 0:
        raise ValueError("没有细胞类型满足 min_cells_per_type 的阈值要求。")

    keep_celltypes = summary["celltype"].tolist()
    df = df[df["celltype"].isin(keep_celltypes)].copy()

    if order_by == "expr_score_desc":
        order = summary.sort_values("expr_score", ascending=False)["celltype"].tolist()
    elif order_by == "mean_desc":
        order = summary.sort_values("mean", ascending=False)["celltype"].tolist()
    elif order_by == "median_desc":
        order = summary.sort_values("median", ascending=False)["celltype"].tolist()
    elif order_by == "pct_nonzero_desc":
        order = summary.sort_values("pct_nonzero", ascending=False)["celltype"].tolist()
    elif order_by == "selected_order":
        order = [x for x in selected_celltypes if x in keep_celltypes]
    else:
        raise ValueError(
            "order_by 只能是：'expr_score_desc', 'mean_desc', 'median_desc', "
            "'pct_nonzero_desc', 或 'selected_order'。"
        )

    summary["celltype"] = pd.Categorical(summary["celltype"], categories=order, ordered=True)
    summary = summary.sort_values("celltype").reset_index(drop=True)

    df["celltype"] = pd.Categorical(df["celltype"], categories=order, ordered=True)
    df = df.sort_values("celltype").reset_index(drop=True)

    fig, ax = plt.subplots(figsize=figsize, dpi=300)

    data_list = [df.loc[df["celltype"] == ct, "expr"].values for ct in order]
    positions = np.arange(1, len(order) + 1)

    bp = ax.boxplot(
        data_list,
        positions=positions,
        widths=0.55,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color="black", linewidth=1.4),
        boxprops=dict(edgecolor="black", linewidth=1.1),
        whiskerprops=dict(color="black", linewidth=1.1),
        capprops=dict(color="black", linewidth=1.1)
    )

    for i, box in enumerate(bp["boxes"]):
        box.set_facecolor(palette[i % len(palette)])

    rng = np.random.default_rng(123)

    for i, ct in enumerate(order):
        y = df.loc[df["celltype"] == ct, "expr"].values

        if len(y) > max_points_per_type:
            y = rng.choice(y, size=max_points_per_type, replace=False)

        x = np.full(len(y), positions[i]) + rng.normal(0, 0.06, len(y))

        ax.scatter(
            x,
            y,
            s=10,
            alpha=0.35,
            color=palette[i % len(palette)],
            edgecolor="none",
            zorder=3
        )

    xtick_labels = []
    for ct in order:
        row = summary.loc[summary["celltype"] == ct].iloc[0]
        n_cells = int(row["n_cells"])
        pct = row["pct_nonzero"]
        xtick_labels.append(f"{ct}\n(n={n_cells}, {pct:.1f}%)")

    ax.set_xticks(positions)
    ax.set_xticklabels(
        xtick_labels,
        rotation=45,
        ha="right",
        fontsize=xtick_fontsize
    )

    ax.set_ylabel(
        f"log1p normalized {gene} expression",
        fontsize=axis_label_fontsize
    )

    ax.set_title(
        f"{gene} expression across selected cell types",
        fontsize=title_fontsize
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.2)
    ax.spines["bottom"].set_linewidth(1.2)

    ax.tick_params(
        axis="both",
        width=1.2,
        length=5,
        labelsize=tick_fontsize
    )

    plt.tight_layout()

    fig.savefig(f"{save_prefix}.pdf", bbox_inches="tight", facecolor="white")
    fig.savefig(f"{save_prefix}.svg", bbox_inches="tight", facecolor="white")
    fig.savefig(f"{save_prefix}.png", bbox_inches="tight", facecolor="white", dpi=600)

    summary.to_csv(f"{save_prefix}_summary.csv", index=False)

    plt.show()

    print("X轴细胞类型排序如下：")
    for i, ct in enumerate(order, start=1):
        row = summary.loc[summary["celltype"] == ct].iloc[0]
        print(
            f"{i}. {ct} | "
            f"mean={row['mean']:.4f}, "
            f"median={row['median']:.4f}, "
            f"pct_nonzero={row['pct_nonzero']:.2f}%, "
            f"expr_score={row['expr_score']:.4f}, "
            f"n_cells={int(row['n_cells'])}"
        )

    return df, summary


immune_celltypes = [
    "Macrophages",
    "CD4 T cells", "CD8 T cells", "CD4 Trm", "CD8 Trm", "Treg", "Tfh", "NK cells", "CD4 T_Cycling",
    "B cells", "B naive", "GCB", "Plasma cells",
    "CTSL_Macrophage", "Alveolar macrophages", "CCSER1_Macrophage", "FYN_macrophages",
    "SPARCL1_Macrophage", "Macrophage_Cycling", "Monocyte",
    "DC1", "DC2", "LAMP3_DC", "Mast cells"
]

plot_df, summary_df = plot_gene_in_selected_celltypes(
    adata=adata,
    gene="CCL5",
    celltype_col="celltype_4_ZZM",
    selected_celltypes=immune_celltypes,
    counts_layer=None,
    target_sum=1e4,
    min_cells_per_type=20,
    order_by="expr_score_desc",
    max_points_per_type=800,
    figsize=(15, 6),
    save_prefix="CCL5_in_immune_celltypes_boxplot_AI_editable",
    title_fontsize=20,
    axis_label_fontsize=17,
    tick_fontsize=14,
    xtick_fontsize=13
)

print(summary_df)